In [2]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import ks_2samp

os.makedirs('../models', exist_ok=True)

df = pd.read_parquet('../data/processed/staged_loans.parquet')

# Populasi untuk Training PD: Hanya pinjaman yang sudah memiliki hasil akhir (Resolved)
resolved_statuses = ['Fully Paid', 'Charged Off', 'Default']
df_train_pop = df[df['loan_status'].isin(resolved_statuses)].copy()

print(f"Total seluruh data bersih : {len(df):,}")
print(f"Populasi Training Model PD : {len(df_train_pop):,}")
print(f"Tingkat Default Historis   : {df_train_pop['is_default'].mean():.2%}")

Total seluruh data bersih : 1,791,760
Populasi Training Model PD : 893,755
Tingkat Default Historis   : 21.48%


In [3]:
def fit_woe_iv(df_train, features, target, bins=5):
    woe_mappings = {}
    iv_summary = []

    for feat in features:
        temp = df_train[[feat, target]].copy()
        
        # Binning kuantil jika fitur numerik, atau gunakan kategori asli jika kategorikal
        if pd.api.types.is_numeric_dtype(temp[feat]):
            temp['bin'], bin_edges = pd.qcut(temp[feat], q=bins, duplicates='drop', retbins=True)
            bin_edges[0] = -np.inf
            bin_edges[-1] = np.inf
        else:
            temp['bin'] = temp[feat].fillna('Missing').astype(str)
            bin_edges = None

        grouped = temp.groupby('bin', observed=False)[target].agg(['count', 'sum'])
        grouped.columns = ['total', 'bad']
        grouped['good'] = grouped['total'] - grouped['bad']

        eps = 1e-6
        grouped['dist_good'] = (grouped['good'] + eps) / grouped['good'].sum()
        grouped['dist_bad'] = (grouped['bad'] + eps) / grouped['bad'].sum()
        grouped['woe'] = np.log(grouped['dist_good'] / grouped['dist_bad'])
        grouped['iv'] = (grouped['dist_good'] - grouped['dist_bad']) * grouped['woe']

        iv_total = grouped['iv'].sum()
        iv_summary.append({'feature': feat, 'IV': round(iv_total, 4)})
        
        woe_mappings[feat] = {
            'woe_dict': grouped['woe'].to_dict(),
            'bin_edges': bin_edges,
            'is_numeric': pd.api.types.is_numeric_dtype(temp[feat])
        }

    return woe_mappings, pd.DataFrame(iv_summary).sort_values(by='IV', ascending=False)

def transform_woe(df_input, woe_mappings):
    df_woe = pd.DataFrame(index=df_input.index)
    for feat, info in woe_mappings.items():
        if info['is_numeric']:
            binned = pd.cut(df_input[feat], bins=info['bin_edges'])
            df_woe[f'{feat}_woe'] = binned.map(info['woe_dict']).astype(float).fillna(0)
        else:
            s = df_input[feat].fillna('Missing').astype(str)
            df_woe[f'{feat}_woe'] = s.map(info['woe_dict']).astype(float).fillna(0)
    return df_woe

# Daftar kandidat fitur risiko kredit
candidate_features = [
    'int_rate', 'grade', 'sub_grade', 'tenor_months', 'fico_range_low',
    'dti', 'annual_inc', 'loan_amnt', 'home_ownership', 'emp_years', 'delinq_2yrs'
]

# Split Train (80%) dan Test (20%) sebelum menghitung WoE agar tidak terjadi data leakage
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df_train_pop[candidate_features], df_train_pop['is_default'],
    test_size=0.2, random_state=42, stratify=df_train_pop['is_default']
)

train_combined = X_train_raw.copy()
train_combined['is_default'] = y_train

woe_maps, iv_df = fit_woe_iv(train_combined, candidate_features, 'is_default', bins=5)
print("=== Hasil Information Value (IV) Seleksi Fitur ===")
print(iv_df.to_string(index=False))

=== Hasil Information Value (IV) Seleksi Fitur ===
       feature     IV
     sub_grade 0.5421
         grade 0.5054
      int_rate 0.4914
fico_range_low 0.1196
           dti 0.0701
home_ownership 0.0407
     loan_amnt 0.0367
    annual_inc 0.0275
     emp_years 0.0081
   delinq_2yrs 0.0021
  tenor_months 0.0000


In [4]:
# Pilih fitur non-multikolinear dengan daya prediksi baik
selected_features = [
    'sub_grade', 'tenor_months', 'fico_range_low', 'dti',
    'annual_inc', 'loan_amnt', 'home_ownership', 'emp_years'
]
selected_woe_maps = {k: woe_maps[k] for k in selected_features}

X_train_woe = transform_woe(X_train_raw, selected_woe_maps)
X_test_woe = transform_woe(X_test_raw, selected_woe_maps)

# Latih model Logistic Regression
pd_model = LogisticRegression(max_iter=1000, random_state=42)
pd_model.fit(X_train_woe, y_train)

# Prediksi probabilitas gagal bayar pada data Test
y_pred_proba = pd_model.predict_proba(X_test_woe)[:, 1]

# Hitung 3 Metrik Utama Validasi Model Kredit Perbankan
auc_score = roc_auc_score(y_test, y_pred_proba)
gini_score = 2 * auc_score - 1
ks_stat, _ = ks_2samp(y_pred_proba[y_test == 1], y_pred_proba[y_test == 0])

print("=== Hasil Evaluasi Model PD (Out-of-Sample Test Set) ===")
print(f"AUC-ROC Score     : {auc_score:.4f}")
print(f"Gini Coefficient  : {gini_score:.4f}  (Standar industri retail/KTA: > 0.35)")
print(f"KS Statistic      : {ks_stat:.4f}  (Standar industri: > 0.25)")

=== Hasil Evaluasi Model PD (Out-of-Sample Test Set) ===
AUC-ROC Score     : 0.7009
Gini Coefficient  : 0.4019  (Standar industri retail/KTA: > 0.35)
KS Statistic      : 0.2918  (Standar industri: > 0.25)


In [5]:
# 1. Transformasi WoE ke seluruh dataset dan prediksi PD kumulatif
X_all_woe = transform_woe(df[selected_features], selected_woe_maps)
df['pd_model_raw'] = pd_model.predict_proba(X_all_woe)[:, 1]

# 2. Konversi Cumulative PD menjadi 12-Month Marginal PD menggunakan pendekatan hazard rate
#    Rumus: PD_12m = 1 - (1 - PD_cum)^(12 / tenor_months)
df['pd_12m'] = 1.0 - (1.0 - df['pd_model_raw']) ** (12.0 / df['tenor_months'])

# Jika nasabah sudah Stage 3 (Macet), paksa PD = 100% (1.0)
df['pd_12m'] = np.where(df['psak71_stage'] == 3, 1.0, df['pd_12m'])

# 3. Estimasi LGD per Grade dari rata-rata historis pinjaman Charged Off
lgd_table = df.groupby('grade')['empirical_lgd'].mean().reset_index()
lgd_table.columns = ['grade', 'segment_lgd']
df = df.merge(lgd_table, on='grade', how='left')
df['segment_lgd'] = df['segment_lgd'].fillna(df['empirical_lgd'].mean())

# 4. Exposure at Default (EAD): Gunakan sisa pokok pinjaman (out_prncp)
#    Catatan: Untuk portofolio aktif, jika out_prncp = 0 pada Stage 3, gunakan funded_amnt
df['ead'] = np.where(
    (df['psak71_stage'] == 3) & (df['out_prncp'] == 0),
    df['funded_amnt'],
    df['out_prncp']
)

# Simpan model, mapping WoE, dan dataset yang sudah memiliki parameter PD, LGD, EAD
joblib.dump({'model': pd_model, 'woe_maps': selected_woe_maps}, '../models/pd_scorecard_model.pkl')
df.to_parquet('../data/processed/portfolio_with_risk_params.parquet', index=False)

print("=== Sampel Parameter Risiko (Stage 1, 2, 3) ===")
print(df[df['psak71_stage'] > 0][['id', 'grade', 'psak71_stage', 'pd_12m', 'segment_lgd', 'ead']].head(10))

=== Sampel Parameter Risiko (Stage 1, 2, 3) ===
          id grade  psak71_stage    pd_12m  segment_lgd       ead
3   66310712     C             1  0.053132     0.940148  15897.65
10  68356421     C             1  0.043333     0.940148   9912.88
11  68426545     C             1  0.048859     0.940148   7080.70
13  66624733     E             3  1.000000     0.930905  18000.00
18  68506798     B             1  0.026185     0.945639   9580.20
25  67849662     C             3  1.000000     0.940148   4225.00
30  67715283     C             3  1.000000     0.940148  16000.00
31  68341789     F             3  1.000000     0.924851  24250.00
33  68415473     C             3  1.000000     0.940148  25000.00
34  68537655     C             1  0.050244     0.940148   7434.43
